In [1]:
import jax
import jax.numpy as jnp
import numpy as onp
import jax_sn
%load_ext autoreload
%autoreload 2
jax.config.update("jax_enable_x64", True)

In [2]:
from jax_sn.tools.setup_test_mgop import setup_slab_mgop, make_xs_test, setup_layered_slab_mgop

In [24]:

def _build_base_mgop():
    """Build the reference mgop with default (Vmap) operators. Expensive — call once."""
    xs = [make_xs_test(m, n_groups=2, l_order=1) for m in ["fuel", "moderator", "reflector"]]
    return setup_layered_slab_mgop(
        dim=3,
        degree=1,
        tn_quadrature_set=3,
        region_sizes=[1.0, 4.0, 2.0],
        region_resolutions=[2,1,3],
        region_xs=xs,
        region_sources=[1.0, 0.5, 0.2],
        transverse_specs=[(2.0, 2), (1.523, 3)],
    )



In [25]:



onp.random.seed(2042026)

base_mgop = _build_base_mgop()

b_apply = jnp.array(onp.random.random_sample(base_mgop.transport_operator.out_structure().shape))
xs = base_mgop.domain.cross_sections_indexed.map_to_full_array(0)


In [26]:
import equinox as eqx

# @eqx.filter_jit
# def sweep_sequential_sa_with_warp(top,sigma_t, x_arr, a : int):    

#     args_flat_ta = _flatten_transport_operator(top)
#     print(args_flat_ta[-6])
#     return sweep_sa_jax(a, *args_flat_ta, sigma_t, x_arr, output_dims = {"result" :  x_arr.shape})[0]

from jax_sn.operators.transport_operator.linear_tetrahedron_warp_kernel import TransportOperatorSequentialWarpSingleAngle, TransportOperatorSequentialWarpMultipleAngles




In [27]:

top_2 = TransportOperatorSequentialWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator[0].topological_sorted_angle, 0)



In [45]:
top_ma = TransportOperatorSequentialWarpMultipleAngles(TransportOperatorSequentialWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator.topological_sorted_angle, 0))

@eqx.filter_jit
def aaa(top_2 : TransportOperatorSequentialWarpSingleAngle, b):
    jax.debug.print("In aaa, top_2 is {top_2} and b is {b}", top_2 = top_2, b = b)
    result =  jnp.sum(top_2.inv(b))
    jax.debug.print("In aaa, top_3 is {top_2} and b is {b}", top_2 = top_2, b = b)
    return result

aaa(top_2, b_apply[0]).block_until_ready()

In aaa, top_2 is TransportOperatorSequentialWarpSingleAngle(
  solution_domain_data=SolutionDomainData(
    domain=Domain(
      vertices=f64[84,3],
      connectivity=i64[216,4],
      face_normals=f64[216,4,3],
      etoe=i64[216,4],
      etof=i64[216,4],
      face_perm_idx=i64[216,4],
      face_template=i64[4,3],
      cross_sections_indexed=CrossSectionsIndexed(
        indices=i64[216],
        cross_section_data=CrossSectionData(
          total=f64[3,2], scattering=f64[3,1,2,2]
        )
      )
    ),
    fem_matrices=FEMMatrices(
      mass_matrix=f64[4,4],
      integration_coeff=f64[4],
      derivative_matrix=f64[3,4,4],
      outflow_matrices=f64[4,4,4],
      inflow_matrices=f64[4,6,4,4,4]
    ),
    precomputed_geometric_data=PrecomputedGeometricData(
      vol_jacobians=f64[216,3,3],
      vol_jacobians_inv=f64[216,3,3],
      vol_jacobians_det=f64[216],
      face_jacobians_det=f64[216,4]
    )
  ),
  topological_sorted_angle=TopologicalSortedAngle(
    levels=i64[2

Array(17437.94645234, dtype=float64)

In [35]:
aaa(top_ma,b_apply).block_until_ready()

Array(1083550.96330294, dtype=float64)

In [36]:
aaa(base_mgop.transport_operator, b_apply)

Array(1083550.96330294, dtype=float64)

In [46]:
grad_f = eqx.filter_jit(eqx.filter_grad(aaa))

In [51]:
grad_f(top_ma, jnp.zeros_like(b_apply)).solution_domain_data

In aaa, top_3 is TransportOperatorSequentialWarpMultipleAngles(
  transport_operator=TransportOperatorSequentialWarpSingleAngle(
    solution_domain_data=SolutionDomainData(
      domain=Domain(
        vertices=f64[84,3],
        connectivity=i64[216,4],
        face_normals=f64[216,4,3],
        etoe=i64[216,4],
        etof=i64[216,4],
        face_perm_idx=i64[216,4],
        face_template=i64[4,3],
        cross_sections_indexed=CrossSectionsIndexed(
          indices=i64[216],
          cross_section_data=CrossSectionData(
            total=f64[3,2], scattering=f64[3,1,2,2]
          )
        )
      ),
      fem_matrices=FEMMatrices(
        mass_matrix=f64[4,4],
        integration_coeff=f64[4],
        derivative_matrix=f64[3,4,4],
        outflow_matrices=f64[4,4,4],
        inflow_matrices=f64[4,6,4,4,4]
      ),
      precomputed_geometric_data=PrecomputedGeometricData(
        vol_jacobians=f64[216,3,3],
        vol_jacobians_inv=f64[216,3,3],
        vol_jacobians_det=f6

EquinoxRuntimeError: Above is the stack outside of JIT. Below is the stack inside of JIT:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/spark-bcdd/jaxvenv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/spark-bcdd/jaxvenv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/spark-bcdd/jaxvenv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/spark-bcdd/jaxvenv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever
    self._run_once()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once
    handle._run()
  File "/usr/lib/python3.12/asyncio/events.py", line 8

In [23]:
from jax_sn.tools import assert_pytree_allclose

assert_pytree_allclose(grad_f(top_ma).solution_domain_data.domain, grad_f(base_mgop.transport_operator).solution_domain_data.domain)#, atol = 1e-6, rtol = 1e-6)

AssertionError: Pytree mismatches found:
(GetAttrKey(name='face_normals'),): 
Not equal to tolerance rtol=1e-12, atol=1e-12

Mismatched elements: 221242 / 575856 (38.4%)
First 5 mismatches are at indices:
 [0, 0, 0]: -615.8051276603946 (ACTUAL), -615.8051276665647 (DESIRED)
 [0, 0, 1]: -1688.1611457442777 (ACTUAL), -1688.1611457509425 (DESIRED)
 [0, 0, 2]: -1752.4413162755955 (ACTUAL), -1752.441316270415 (DESIRED)
 [1, 0, 0]: -811.5424270442454 (ACTUAL), -811.5424270459916 (DESIRED)
 [1, 3, 2]: -2766.2596024627564 (ACTUAL), -2766.2596024682425 (DESIRED)
Max absolute difference among violations: 8.19563866e-06
Max relative difference among violations: 2.68838234e-07
 ACTUAL: array([[[-6.158051e+02, -1.688161e+03, -1.752441e+03],
        [ 1.161027e+04,  2.560793e+04,  3.980670e+04],
        [ 7.185240e+03,  4.722481e+03,  2.102177e+04],...
 DESIRED: array([[[-6.158051e+02, -1.688161e+03, -1.752441e+03],
        [ 1.161027e+04,  2.560793e+04,  3.980670e+04],
        [ 7.185240e+03,  4.72

In [32]:
%timeit top_2.inv(b_apply[0]).block_until_ready()

525 ms ± 921 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
onp.testing.assert_allclose(
    top_2.inv(b_apply[0]),
    base_mgop.transport_operator[0].inv(b_apply[0]),
    rtol=1e-5,
    atol=1e-5,)

In [25]:
onp.testing.assert_allclose(
    _sweep_sequential_ma_with_warp(base_mgop.transport_operator, b_apply),
    base_mgop.transport_operator.inv(b_apply),
    rtol=1e-5,
    atol=1e-5,)

In [ ]:
from warp.jax_experimental import jax_callable

jax_callable()

In [ ]:
result_buffer = jnp.zeros_like(b_apply[0])

In [ ]:
ws.single_angle(0, xs.total, b_apply[0], result_buffer)

In [ ]:
result_buffer_ma = jnp.zeros_like(b_apply)

In [ ]:
import warp as wp
reee = (ws.multi_angle(xs.total, b_apply, result_buffer_ma))

In [ ]:
%timeit -n 10 -r 3 ws.single_angle(0, xs.total, b_apply[0], result_buffer)

523 ms ± 382 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [ ]:
%timeit -n 5 -r 3 ws.multi_angle(xs.total, b_apply, result_buffer_ma)

671 ms ± 10.5 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [ ]:
%timeit  base_mgop.transport_operator.inv(b_apply).block_until_ready()

709 ms ± 4.18 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
onp.testing.assert_allclose(result, base_mgop.transport_operator[0].inv(b_apply[0]))

In [ ]:
onp.testing.assert_allclose(result_buffer_ma, base_mgop.transport_operator.inv(b_apply))

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=0

Mismatched elements: 1 / 4188672 (2.39e-05%)
Mismatch at index:
 [58, 12959, 0]: -0.00039902634642562795 (ACTUAL), -0.0003990261440360854 (DESIRED)
Max absolute difference among violations: 2.02389543e-10
Max relative difference among violations: 5.07208727e-07
 ACTUAL: array([[[ 6.439454e+03,  6.434124e+03,  6.379089e+03,  6.690254e+03],
        [ 6.440841e+03,  6.396528e+03,  6.682499e+03,  6.633334e+03],
        [ 6.430937e+03,  6.705093e+03,  6.635500e+03,  6.688554e+03],...
 DESIRED: array([[[ 6.439454e+03,  6.434124e+03,  6.379089e+03,  6.690254e+03],
        [ 6.440841e+03,  6.396528e+03,  6.682499e+03,  6.633334e+03],
        [ 6.430937e+03,  6.705093e+03,  6.635500e+03,  6.688554e+03],...

In [ ]:
import pyvista as pv
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
plotter  = pv.Plotter()
plotter.add_mesh(mesh_to_pyvista_mesh(mgop.domain.vertices, mgop.domain.connectivity), scalars = mgop.domain.cross_sections_indexed.indices)
plotter.show()

Widget(value='<iframe src="http://localhost:44001/index.html?ui=P_0xe874841174a0_5&reconnect=auto" class="pyvi…